# 04. Final Coloring Book Pipeline

? ???? 01, 02, 03? ????? ??? ??? ??? ?????.

??:

- `data/flowers.jpg` ?? ???
- ??? ?? ?? K
- ??? Canny ???
- ? ??
- ?? ?? ??

??:

- ?? ???
- K-Means, Posterization, Median Cut ?? ??? ??
- ?? ??? ?? ??? ? ??
- Sobel, Laplacian, Canny, Hybrid Color Boundary ??? ??
- Connected Components ?? ?? ??
- ?? ??? ??? ?? ???? ???
- RGB ???? ?? ??? ?
- K ??? ?? ??? ??


In [ ]:
# Kernel -> Restart & Run All ? ???? ?????.
import os
import sys
import time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

# ???? project root ?? server ?? ???? ???? server/data? server/src? ????.
CURRENT_DIR = Path.cwd()
FALLBACK_SERVER_DIR = Path(r"C:\Users\qhtm0\Desktop\project\HCI_PROJECT\server")
if (CURRENT_DIR / "data").is_dir() and (CURRENT_DIR / "src").is_dir():
    SERVER_DIR = CURRENT_DIR
elif (CURRENT_DIR / "server" / "data").is_dir() and (CURRENT_DIR / "server" / "src").is_dir():
    SERVER_DIR = CURRENT_DIR / "server"
else:
    SERVER_DIR = FALLBACK_SERVER_DIR

SRC_DIR = SERVER_DIR / "src"
sys.path.insert(0, str(SRC_DIR))

import importlib
import coloringbook_utils as coloringbook_utils
importlib.reload(coloringbook_utils)
from coloringbook_utils import *

# ?? ?? ??? ?? ?? ??? ?????.
ensure_dirs()
OUTPUT_DIR = str(SERVER_DIR / "ychoutput")
os.makedirs(OUTPUT_DIR, exist_ok=True)
plt.rcParams["figure.dpi"] = 120


def load_image_preserve_size(image_path):
    """Load an RGB image without resizing so result PNGs keep the source size."""
    bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(f"Image not found or unsupported: {image_path}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


# ??? ?? ???? ??? ?? ?????? ?????.
IMAGE_PATH = str(SERVER_DIR / "data" / "ych.jpg")
if not IMAGE_PATH or not os.path.exists(IMAGE_PATH):
    print(f"WARNING: '{IMAGE_PATH}' not found, using built-in sample image instead.")
    IMAGE_PATH = None

try:
    image = load_image_preserve_size(IMAGE_PATH)
except FileNotFoundError as exc:
    print(f"WARNING: {exc}; using built-in sample image instead.")
    image = load_image(None, max_size=100000)

show_images([("Original Image", image)], cols=1, figsize=(5, 5))


## ?? ????

?? ????? K=5, 10, 20? ????, ?? ??? ?? K=10?? ???? ??? ????.


In [ ]:
# K is selected automatically from the smoothed image color groups.
# Change DIFFICULTY to "??", "??", or "???".
DIFFICULTY = "\uBCF4\uD1B5"
DIFFICULTY_K_MARGIN = {"\uC26C\uC6C0": 0, "\uBCF4\uD1B5": 10, "\uC5B4\uB824\uC6C0": 20}
MIN_K_LOWER_BOUND = 3
MIN_K_UPPER_BOUND = 40
K_UPPER_BOUND = 60
K_VALUES = None

# Canny ??????. ?? ??? ???? ??? ??? ??? ??? ? ????.
CANNY_LOW = 80
CANNY_HIGH = 180

# ?? Hybrid edge? K-Means ?? ?? ??? ??? Lab ? ??? ? ??? ?????.
# ?? ???? ???/??????? ?? ??? edge? ?????.
COLOR_EDGE_DELTA = 35
FINAL_CLOSE_ITER = 4

# ?? ?? edge ?? ?????. ?? ? ??? ??? ?? ?? ????.
OBJECT_MIN_AREA = 260
OBJECT_CLOSE_KERNEL = 5

# ???, ?, ?? ?? ?? ???? ?? ???? ?? ?? ?? ???? ????.
DETAIL_CANNY_LOW = 90
DETAIL_CANNY_HIGH = 200
DETAIL_MIN_AREA = 3
DETAIL_MAX_AREA = 1200
DARK_DETAIL_THRESHOLD = 70
DARK_DETAIL_MIN_AREA = 5
DARK_DETAIL_MAX_AREA = 900
SEGMENTATION_DARK_REGION_MIN_AREA = 35
SEGMENTATION_DARK_REGION_MAX_AREA = 1400
SEGMENTATION_DARK_REGION_MIN_CIRCULARITY = 0.28
SEGMENTATION_DARK_REGION_MIN_EXTENT = 0.22
SEGMENTATION_DARK_REGION_MAX_ASPECT = 2.4
SEGMENTATION_DETAIL_CANNY_LOW = 35
SEGMENTATION_DETAIL_CANNY_HIGH = 95
SEGMENTATION_DETAIL_MIN_PIXELS = 10
SEGMENTATION_DETAIL_MAX_PIXELS = 1800
SEGMENTATION_DETAIL_MIN_SPAN = 28
SEGMENTATION_DETAIL_MAX_SPAN = 460
SEGMENTATION_DETAIL_NON_DARK_THRESHOLD = 82
SEGMENTATION_CLOSED_DETAIL_LOW = 45
SEGMENTATION_CLOSED_DETAIL_HIGH = 130
SEGMENTATION_CLOSED_DETAIL_MIN_PIXELS = 8
SEGMENTATION_CLOSED_DETAIL_MAX_PIXELS = 1800
SEGMENTATION_CLOSED_DETAIL_MIN_CONTOUR_AREA = 18
SEGMENTATION_CLOSED_DETAIL_MAX_CONTOUR_AREA = 9000
SEGMENTATION_CLOSED_DETAIL_MAX_SPAN = 220
SEGMENTATION_CLOSED_DETAIL_MIN_EXTENT = 0.06
SEGMENTATION_CLOSED_DETAIL_CLOSE_KERNEL = 5

# ?? ???? ? ?????. ?? ?? ?? ?????.
LINE_THICKNESS = 1

# 02? ???? ?? ??? Hybrid ? ?????.
EDGE_COMPARE_THICKNESS = 1

# 02_edge_detection.ipynb에서 보낸 adaptive threshold contour 방식은 비교용으로 함께 평가합니다.
# 최종 segmentation에는 닫힌 색 영역이 더 안정적인 Object-first contour를 사용합니다.
ADAPTIVE_BLOCK_SIZE = 15
ADAPTIVE_C = 4
ADAPTIVE_BILATERAL_DIAMETER = 9
ADAPTIVE_BILATERAL_SIGMA = 75

# ?? ?? ??? ??? ?? ??? ??? ?????.
MIN_REGION_AREA = 160

# ??? ?? ?? ?? ??? segmentation ?? ???? ??? ?? ??? ? ?? ?????.
SEGMENTATION_OUTPUT_SCALE = 2.0

# K-Means 이전에 질감/음영을 먼저 색 덩어리로 평탄화합니다.
# Mean Shift 이후 Bilateral filter를 적용해 작은 색 변화가 군집을 깨지 않게 합니다.
PRE_KMEANS_MEAN_SHIFT_ENABLED = True
MEAN_SHIFT_SPATIAL_RADIUS = 12
MEAN_SHIFT_COLOR_RADIUS = 28
SIMPLIFY_DIAMETER = 9
SIMPLIFY_SIGMA_COLOR = 90
SIMPLIFY_SIGMA_SPACE = 90


## ?? ????? ??


In [ ]:
def smooth_before_kmeans(image):
    """Smooth texture and soft shading before K-Means color clustering."""
    if PRE_KMEANS_MEAN_SHIFT_ENABLED:
        bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        shifted = cv2.pyrMeanShiftFiltering(
            bgr,
            sp=MEAN_SHIFT_SPATIAL_RADIUS,
            sr=MEAN_SHIFT_COLOR_RADIUS,
        )
        image = cv2.cvtColor(shifted, cv2.COLOR_BGR2RGB)
    return cv2.bilateralFilter(
        image,
        SIMPLIFY_DIAMETER,
        SIMPLIFY_SIGMA_COLOR,
        SIMPLIFY_SIGMA_SPACE,
    )


def estimate_min_k_from_smoothed_colors(image, max_sample=40000, rgb_bin_size=16):
    """Estimate the minimum useful K from coarse color groups in the smoothed image."""
    pixels = image.reshape(-1, 3)
    if len(pixels) > max_sample:
        rng = np.random.default_rng(7)
        pixels = pixels[rng.choice(len(pixels), max_sample, replace=False)]

    coarse_rgb = (pixels.astype(np.uint16) // rgb_bin_size).astype(np.uint16)
    coarse_unique_colors = int(len(np.unique(coarse_rgb, axis=0)))

    lab = cv2.cvtColor(pixels.reshape(1, -1, 3).astype(np.uint8), cv2.COLOR_RGB2LAB).astype(np.float32)[0]
    lab_spread = float(np.mean(np.linalg.norm(lab - lab.mean(axis=0), axis=1)))

    color_group_score = np.log2(max(coarse_unique_colors, 2))
    estimated = int(np.ceil(0.9 * color_group_score + lab_spread / 14.0))
    min_k = int(np.clip(estimated, MIN_K_LOWER_BOUND, MIN_K_UPPER_BOUND))
    return min_k, {
        "coarse_unique_colors": coarse_unique_colors,
        "lab_spread": lab_spread,
        "estimated_min_k": min_k,
    }


def difficulty_k_values(min_k):
    """Build K values for easy/normal/hard modes from the estimated minimum K."""
    values = {}
    for name, margin in DIFFICULTY_K_MARGIN.items():
        values[name] = int(np.clip(min_k + margin, MIN_K_LOWER_BOUND, K_UPPER_BOUND))
    return values


def upscale_for_segmentation_output(image, simplified, quantized, labels, palette, scale):
    """Upscale the canvas before segmentation and final number placement."""
    scale = float(scale)
    if scale <= 1.0:
        return image, simplified, quantized, labels
    h, w = labels.shape[:2]
    size = (int(round(w * scale)), int(round(h * scale)))
    up_image = cv2.resize(image, size, interpolation=cv2.INTER_CUBIC)
    up_simplified = cv2.resize(simplified, size, interpolation=cv2.INTER_CUBIC)
    up_labels = cv2.resize(labels.astype(np.int32), size, interpolation=cv2.INTER_NEAREST).astype(np.int32)
    up_quantized = palette[up_labels]
    return up_image, up_simplified, up_quantized, up_labels


def resize_preview_image(image, target_shape):
    """Resize comparison-only images to the final output canvas size."""
    h, w = target_shape[:2]
    interpolation = cv2.INTER_NEAREST if count_unique_colors(image) <= 256 else cv2.INTER_CUBIC
    return cv2.resize(image, (w, h), interpolation=interpolation)


# 0. K-Means ?? ???: ??, ??, ?????? ?? ?????.
(simplified_image, simplify_time) = timed_call(smooth_before_kmeans, image)
MIN_RECOMMENDED_K, k_recommendation_info = estimate_min_k_from_smoothed_colors(simplified_image)
DIFFICULTY_K_VALUES = difficulty_k_values(MIN_RECOMMENDED_K)
if DIFFICULTY not in DIFFICULTY_K_VALUES:
    raise ValueError(f"DIFFICULTY must be one of {list(DIFFICULTY_K_VALUES)}, got {DIFFICULTY!r}")
K = DIFFICULTY_K_VALUES[DIFFICULTY]
K_VALUES = list(DIFFICULTY_K_VALUES.values())

# 1. ?? ???: ???? ???? ???? 01? ???? ? ????? ?????.
(kmeans_img, kmeans_palette, kmeans_labels), kmeans_time = timed_call(kmeans_quantization_with_labels, simplified_image, K)
(poster_img, poster_palette), poster_time = timed_call(posterization, simplified_image, K)
(median_img, median_palette), median_time = timed_call(median_cut_quantization, simplified_image, K)

quality_rows = []
for name, result, runtime in [
    ("K-Means", kmeans_img, kmeans_time),
    ("Posterization", poster_img, poster_time),
    ("Median Cut", median_img, median_time),
]:
    quality_rows.append({
        "algorithm": name,
        "runtime_sec": runtime,
        "color_error": quantization_error(simplified_image, result),
        "unique_colors": count_unique_colors(result),
    })

base_image_shape = image.shape
base_image_for_variants = image.copy()
base_simplified_for_variants = simplified_image.copy()
base_simplified_color_count = count_unique_colors(simplified_image)
image, simplified_image, kmeans_img, kmeans_labels = upscale_for_segmentation_output(
    image,
    simplified_image,
    kmeans_img,
    kmeans_labels,
    kmeans_palette,
    SEGMENTATION_OUTPUT_SCALE,
)
poster_img = resize_preview_image(poster_img, image.shape)
median_img = resize_preview_image(median_img, image.shape)
output_scale_info = {
    "scale": SEGMENTATION_OUTPUT_SCALE,
    "base_shape": base_image_shape[:2],
    "output_shape": image.shape[:2],
}
AREA_SCALE_FOR_OUTPUT = max(1.0, float(SEGMENTATION_OUTPUT_SCALE) ** 2)
OBJECT_MIN_AREA_FOR_OUTPUT = int(round(OBJECT_MIN_AREA * AREA_SCALE_FOR_OUTPUT))
MIN_REGION_AREA_FOR_OUTPUT = int(round(MIN_REGION_AREA * AREA_SCALE_FOR_OUTPUT))
DETAIL_MIN_AREA_FOR_OUTPUT = int(round(DETAIL_MIN_AREA * AREA_SCALE_FOR_OUTPUT))
DETAIL_MAX_AREA_FOR_OUTPUT = int(round(DETAIL_MAX_AREA * AREA_SCALE_FOR_OUTPUT))
DARK_DETAIL_MIN_AREA_FOR_OUTPUT = int(round(DARK_DETAIL_MIN_AREA * AREA_SCALE_FOR_OUTPUT))
DARK_DETAIL_MAX_AREA_FOR_OUTPUT = int(round(DARK_DETAIL_MAX_AREA * AREA_SCALE_FOR_OUTPUT))
SEGMENTATION_DARK_REGION_MIN_AREA_FOR_OUTPUT = int(round(SEGMENTATION_DARK_REGION_MIN_AREA * AREA_SCALE_FOR_OUTPUT))
SEGMENTATION_DARK_REGION_MAX_AREA_FOR_OUTPUT = int(round(SEGMENTATION_DARK_REGION_MAX_AREA * AREA_SCALE_FOR_OUTPUT))

def object_first_edges(label_map, min_area=260, close_kernel=5, thickness=1):
    """Create closed coloring-book edges from cleaned color-object contours."""
    labels = np.asarray(label_map, dtype=np.int32)
    edge = np.zeros(labels.shape, dtype=np.uint8)
    kernel = np.ones((close_kernel, close_kernel), np.uint8)

    for label_id in np.unique(labels):
        mask = (labels == label_id).astype(np.uint8) * 255
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8), iterations=1)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            if cv2.contourArea(contour) < min_area:
                continue
            cv2.drawContours(edge, [contour], -1, 255, thickness, cv2.LINE_8)

    return edge

def thin_binary_edges(edges):
    """Thin binary edge strokes with Zhang-Suen thinning."""
    img = (edges > 0).astype(np.uint8)
    changed = True
    while changed:
        changed = False
        for step in (0, 1):
            padded = np.pad(img, 1, mode="constant")
            marker = np.zeros_like(img)
            for y in range(1, padded.shape[0] - 1):
                for x in range(1, padded.shape[1] - 1):
                    if padded[y, x] == 0:
                        continue
                    p2 = padded[y - 1, x]
                    p3 = padded[y - 1, x + 1]
                    p4 = padded[y, x + 1]
                    p5 = padded[y + 1, x + 1]
                    p6 = padded[y + 1, x]
                    p7 = padded[y + 1, x - 1]
                    p8 = padded[y, x - 1]
                    p9 = padded[y - 1, x - 1]
                    neighbors = [p2, p3, p4, p5, p6, p7, p8, p9]
                    count = sum(neighbors)
                    transitions = sum((neighbors[i] == 0 and neighbors[(i + 1) % 8] == 1) for i in range(8))
                    if not (2 <= count <= 6 and transitions == 1):
                        continue
                    if step == 0:
                        if p2 * p4 * p6 == 0 and p4 * p6 * p8 == 0:
                            marker[y - 1, x - 1] = 1
                    else:
                        if p2 * p4 * p8 == 0 and p2 * p6 * p8 == 0:
                            marker[y - 1, x - 1] = 1
            if np.any(marker):
                img[marker == 1] = 0
                changed = True
    return (img * 255).astype(np.uint8)

def detail_expression_edges(
    image,
    object_edges,
    low=100,
    high=220,
    min_area=8,
    max_area=450,
    dark_threshold=70,
    dark_min_area=5,
    dark_max_area=900,
):
    """Extract fine expression/detail strokes without affecting segmentation regions."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    # Thin edge details such as mouth curves and small wrinkles.
    detail = cv2.Canny(gray, low, high)
    detail = cv2.bitwise_and(detail, cv2.bitwise_not(object_edges))
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(detail, 8)
    filtered_edges = np.zeros_like(detail)
    for i in range(1, n_labels):
        area = int(stats[i, cv2.CC_STAT_AREA])
        if min_area <= area <= max_area:
            filtered_edges[labels == i] = 255

    # Dark filled details such as pupils, mouths, bats, and tiny facial marks.
    dark = (gray < dark_threshold).astype(np.uint8) * 255
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(dark, 8)
    filtered_dark = np.zeros_like(dark)
    for i in range(1, n_labels):
        area = int(stats[i, cv2.CC_STAT_AREA])
        if dark_min_area <= area <= dark_max_area:
            filtered_dark[labels == i] = 255

    return cv2.bitwise_or(filtered_edges, filtered_dark)

def dark_detail_region_edges(
    image,
    dark_threshold=70,
    min_area=35,
    max_area=1400,
    thickness=1,
):
    """Promote compact dark facial details, such as eyes, to segmentation contours."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    dark = (gray < dark_threshold).astype(np.uint8) * 255
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(dark, 8)
    promoted = np.zeros_like(dark)

    for component_id in range(1, n_labels):
        area = int(stats[component_id, cv2.CC_STAT_AREA])
        if not (min_area <= area <= max_area):
            continue

        x = int(stats[component_id, cv2.CC_STAT_LEFT])
        y = int(stats[component_id, cv2.CC_STAT_TOP])
        w = int(stats[component_id, cv2.CC_STAT_WIDTH])
        h = int(stats[component_id, cv2.CC_STAT_HEIGHT])
        if w == 0 or h == 0:
            continue

        aspect = max(w / h, h / w)
        extent = area / float(w * h)
        if aspect > SEGMENTATION_DARK_REGION_MAX_ASPECT or extent < SEGMENTATION_DARK_REGION_MIN_EXTENT:
            continue

        component = (labels[y:y + h, x:x + w] == component_id).astype(np.uint8) * 255
        contours, _ = cv2.findContours(component, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            contour_area = cv2.contourArea(contour)
            perimeter = cv2.arcLength(contour, True)
            if perimeter <= 0:
                continue
            circularity = 4.0 * np.pi * contour_area / (perimeter * perimeter)
            if circularity < SEGMENTATION_DARK_REGION_MIN_CIRCULARITY:
                continue

            contour = contour + np.array([[[x, y]]], dtype=contour.dtype)
            cv2.drawContours(promoted, [contour], -1, 255, thickness, cv2.LINE_8)

    return promoted


def source_detail_boundary_edges(
    image,
    object_edges,
    low=35,
    high=95,
    min_pixels=10,
    max_pixels=1800,
    min_span=28,
    max_span=460,
    non_dark_threshold=82,
):
    """Recover broad internal shade boundaries that low-K color labels merge away."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    edges = cv2.Canny(gray, low, high)
    for channel in cv2.split(lab):
        edges = cv2.bitwise_or(edges, cv2.Canny(channel, low, high))

    edges = cv2.bitwise_and(edges, (gray > non_dark_threshold).astype(np.uint8) * 255)
    edges = cv2.bitwise_and(edges, cv2.bitwise_not(cv2.dilate(object_edges, np.ones((3, 3), np.uint8), iterations=1)))

    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(edges, 8)
    promoted = np.zeros_like(edges)
    for component_id in range(1, n_labels):
        pixels = int(stats[component_id, cv2.CC_STAT_AREA])
        if not (min_pixels <= pixels <= max_pixels):
            continue

        x = int(stats[component_id, cv2.CC_STAT_LEFT])
        y = int(stats[component_id, cv2.CC_STAT_TOP])
        w = int(stats[component_id, cv2.CC_STAT_WIDTH])
        h = int(stats[component_id, cv2.CC_STAT_HEIGHT])
        span = max(w, h)
        if not (min_span <= span <= max_span):
            continue

        component = (labels[y:y + h, x:x + w] == component_id).astype(np.uint8) * 255
        contours, _ = cv2.findContours(component, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        for contour in contours:
            arc = cv2.arcLength(contour, False)
            if arc < min_span:
                continue
            contour = contour + np.array([[[x, y]]], dtype=contour.dtype)
            cv2.drawContours(promoted, [contour], -1, 255, 1, cv2.LINE_8)

    return promoted


def closed_detail_shape_edges(
    image,
    object_edges,
    low=SEGMENTATION_CLOSED_DETAIL_LOW,
    high=SEGMENTATION_CLOSED_DETAIL_HIGH,
    min_pixels=SEGMENTATION_CLOSED_DETAIL_MIN_PIXELS,
    max_pixels=SEGMENTATION_CLOSED_DETAIL_MAX_PIXELS,
    min_contour_area=SEGMENTATION_CLOSED_DETAIL_MIN_CONTOUR_AREA,
    max_contour_area=SEGMENTATION_CLOSED_DETAIL_MAX_CONTOUR_AREA,
    max_span=SEGMENTATION_CLOSED_DETAIL_MAX_SPAN,
    min_extent=SEGMENTATION_CLOSED_DETAIL_MIN_EXTENT,
    close_kernel=SEGMENTATION_CLOSED_DETAIL_CLOSE_KERNEL,
):
    """Promote closed or nearly closed detail shapes to segmentation boundaries."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    edges = cv2.Canny(gray, low, high)
    for channel in cv2.split(lab):
        edges = cv2.bitwise_or(edges, cv2.Canny(channel, low, high))

    object_guard = cv2.dilate(object_edges, np.ones((3, 3), np.uint8), iterations=1)
    edges = cv2.bitwise_and(edges, cv2.bitwise_not(object_guard))

    kernel_size = max(3, int(close_kernel))
    if kernel_size % 2 == 0:
        kernel_size += 1
    close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    closed_edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, close, iterations=1)

    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(closed_edges, 8)
    promoted = np.zeros_like(edges)
    for component_id in range(1, n_labels):
        pixels = int(stats[component_id, cv2.CC_STAT_AREA])
        if not (min_pixels <= pixels <= max_pixels):
            continue
        x = int(stats[component_id, cv2.CC_STAT_LEFT])
        y = int(stats[component_id, cv2.CC_STAT_TOP])
        w = int(stats[component_id, cv2.CC_STAT_WIDTH])
        h = int(stats[component_id, cv2.CC_STAT_HEIGHT])
        if max(w, h) > max_span:
            continue
        extent = pixels / max(1, w * h)
        if extent < min_extent:
            continue

        component = (labels[y:y + h, x:x + w] == component_id).astype(np.uint8) * 255
        contours, _ = cv2.findContours(component, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            contour_area = cv2.contourArea(contour)
            if not (min_contour_area <= contour_area <= max_contour_area):
                continue
            contour = contour + np.array([[[x, y]]], dtype=contour.dtype)
            cv2.drawContours(promoted, [contour], -1, 255, 1, cv2.LINE_8)
    return promoted


def border_connected_region_ids(region_map):
    """Return connected-component ids touching the image border; these are page background."""
    if region_map.size == 0:
        return set()
    border_ids = np.concatenate([
        region_map[0, :],
        region_map[-1, :],
        region_map[:, 0],
        region_map[:, -1],
    ])
    return set(int(v) for v in np.unique(border_ids) if int(v) > 0)


def best_label_point(region_map, region_id, bbox):
    x, y, w, h = bbox
    component = (region_map[y:y + h, x:x + w] == region_id).astype(np.uint8)
    if component.size == 0 or np.count_nonzero(component) == 0:
        return x + w / 2, y + h / 2
    dist = cv2.distanceTransform(component, cv2.DIST_L2, 5)
    _, _, _, max_loc = cv2.minMaxLoc(dist)
    return x + max_loc[0], y + max_loc[1]


def main_object_mask_from_regions(region_map, regions, background_ids, source_image=None, bridge_iterations=3):
    """Keep the main subject silhouette so isolated paper/background texture is ignored."""
    if source_image is not None:
        border = np.concatenate([
            source_image[0, :, :],
            source_image[-1, :, :],
            source_image[:, 0, :],
            source_image[:, -1, :],
        ], axis=0)
        bg = np.median(border.reshape(-1, 3), axis=0)
        diff = np.linalg.norm(source_image.astype(np.float32) - bg.astype(np.float32), axis=2)
        gray_src = cv2.cvtColor(source_image, cv2.COLOR_RGB2GRAY)
        seed = ((diff > 28) | (gray_src < 120)).astype(np.uint8) * 255
        seed = cv2.morphologyEx(seed, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8), iterations=1)
        seed = cv2.morphologyEx(seed, cv2.MORPH_CLOSE, np.ones((17, 17), np.uint8), iterations=2)
        seed = cv2.dilate(seed, np.ones((5, 5), np.uint8), iterations=2)

        n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(seed, 8)
        if n_labels > 1:
            largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
            mask = (labels == largest).astype(np.uint8) * 255
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            filled = np.zeros_like(mask)
            if contours:
                cv2.drawContours(filled, contours, -1, 255, -1)
                filled = cv2.morphologyEx(filled, cv2.MORPH_CLOSE, np.ones((21, 21), np.uint8), iterations=1)
                return filled

    candidate = np.zeros(region_map.shape, dtype=np.uint8)
    for region in regions:
        region_id = int(region["id"])
        if region_id in background_ids:
            continue
        candidate[region_map == region_id] = 255

    if np.count_nonzero(candidate) == 0:
        return candidate

    bridge = cv2.dilate(candidate, np.ones((3, 3), np.uint8), iterations=bridge_iterations)
    bridge = cv2.morphologyEx(bridge, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8), iterations=1)
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(bridge, 8)
    if n_labels <= 1:
        return bridge

    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    mask = (labels == largest).astype(np.uint8) * 255
    mask = cv2.dilate(mask, np.ones((5, 5), np.uint8), iterations=1)
    return mask

def fit_number_font_scale(text, bbox, max_scale=0.82, min_scale=0.22, padding=5):
    """Choose a readable number size that still fits inside the region bbox."""
    _, _, w, h = bbox
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = max_scale
    while scale > min_scale:
        (tw, th), _ = cv2.getTextSize(text, font, scale, 1)
        if tw <= max(1, w - padding * 2) and th <= max(1, h - padding * 2):
            return scale
        scale -= 0.04
    return min_scale


def region_boundary_mask(region_map, min_component_area=10):
    """Return a single shared boundary mask from a region map without per-region double contours."""
    regions_arr = np.asarray(region_map, dtype=np.int32)
    boundary = np.zeros(regions_arr.shape, dtype=np.uint8)
    horizontal = regions_arr[:, 1:] != regions_arr[:, :-1]
    vertical = regions_arr[1:, :] != regions_arr[:-1, :]
    boundary[:, 1:][horizontal] = 255
    boundary[1:, :][vertical] = 255
    boundary[regions_arr == 0] = 0

    if min_component_area > 1:
        n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(boundary, 8)
        filtered = np.zeros_like(boundary)
        for component_id in range(1, n_labels):
            if stats[component_id, cv2.CC_STAT_AREA] >= min_component_area:
                filtered[labels == component_id] = 255
        boundary = filtered
    return boundary


def draw_simplified_boundary_layer(canvas, region_map, line_color=(255, 0, 0), epsilon_ratio=0.004):
    """Draw one simplified line for each shared segmentation boundary component."""
    boundary = region_boundary_mask(region_map)
    contours, _ = cv2.findContours(boundary, cv2.RETR_LIST, cv2.CHAIN_APPROX_NONE)
    for contour in contours:
        if len(contour) < 6:
            continue
        arc = cv2.arcLength(contour, False)
        if arc < 8:
            continue
        epsilon = max(1.2, epsilon_ratio * arc)
        simplified = cv2.approxPolyDP(contour, epsilon, False)
        cv2.polylines(canvas, [simplified], False, line_color, 1, cv2.LINE_AA)
    return canvas


def draw_paint_by_number_style(
    line_image,
    detail_edges,
    regions,
    region_map,
    background_ids,
    source_image=None,
    line_color=(0, 0, 0),
    detail_gray=150,
    text_gray=145,
    font_scale=0.62,
    segmentation_edges=None,
):
    """Render a PDF-like paint-by-number page with single shared boundaries and fitted numbers."""
    h, w = region_map.shape[:2]
    canvas = np.full((h, w, 3), 255, dtype=np.uint8)

    # Draw the exact segmentation mask used for connected-component regions so
    # printed color numbers match the visible coloring boundaries.
    if segmentation_edges is None:
        draw_simplified_boundary_layer(canvas, region_map, line_color=line_color)
    else:
        canvas[segmentation_edges > 0] = line_color

    # Detail strokes stay separate from segmentation boundaries: same 1px width,
    # much lighter gray, so they are visible without becoming color-region borders.
    if detail_edges is not None:
        detail = clean_edges(detail_edges, open_iter=0, close_iter=0, thickness=1)
        detail_contours, _ = cv2.findContours(detail, cv2.RETR_LIST, cv2.CHAIN_APPROX_TC89_KCOS)
        for contour in detail_contours:
            if cv2.contourArea(contour) < 2:
                continue
            cv2.drawContours(canvas, [contour], -1, (detail_gray, detail_gray, detail_gray), 1, cv2.LINE_AA)

    font = cv2.FONT_HERSHEY_SIMPLEX
    for region in sorted(regions, key=lambda r: r["area"], reverse=True):
        if region["area"] < MIN_REGION_AREA_FOR_OUTPUT:
            continue
        text = str(region.get("color_id", region["id"]))
        cx, cy = best_label_point(region_map, int(region["id"]), region["bbox"])
        scale = min(font_scale, fit_number_font_scale(text, region["bbox"]))
        thickness = 1
        (tw, th), base = cv2.getTextSize(text, font, scale, thickness)
        x0, y0, bw, bh = region["bbox"]
        x = int(np.clip(cx - tw / 2, x0 + 2, max(x0 + 2, x0 + bw - tw - 2)))
        y = int(np.clip(cy + th / 2, y0 + th + 2, max(y0 + th + 2, y0 + bh - 2)))
        cv2.putText(canvas, text, (x, y), font, scale, (text_gray, text_gray, text_gray), thickness, cv2.LINE_AA)
    return canvas


def render_segmentation_only(region_map, regions, line_color=(190, 190, 190)):
    """Render only single shared color-region boundaries for inspecting segmentation."""
    h, w = region_map.shape[:2]
    canvas = np.full((h, w, 3), 255, dtype=np.uint8)
    return draw_simplified_boundary_layer(canvas, region_map, line_color=line_color)


def render_detail_only(detail_edges, detail_gray=150):
    """Render only expression/detail strokes as a separate light-gray layer."""
    canvas = np.full((*detail_edges.shape[:2], 3), 255, dtype=np.uint8)
    detail = clean_edges(detail_edges, open_iter=0, close_iter=0, thickness=1)
    detail_contours, _ = cv2.findContours(detail, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_TC89_KCOS)
    for contour in detail_contours:
        if cv2.contourArea(contour) < 2:
            continue
        cv2.drawContours(canvas, [contour], -1, (detail_gray, detail_gray, detail_gray), 1, cv2.LINE_AA)
    return canvas


def adaptive_threshold_contour_edges(image):
    """Reproduce the sent 02_edge_detection contour algorithm without fixed resizing."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    smooth = cv2.bilateralFilter(
        gray,
        ADAPTIVE_BILATERAL_DIAMETER,
        ADAPTIVE_BILATERAL_SIGMA,
        ADAPTIVE_BILATERAL_SIGMA,
    )
    bw = cv2.adaptiveThreshold(
        smooth,
        255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        ADAPTIVE_BLOCK_SIZE,
        ADAPTIVE_C,
    )
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    bw = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel, iterations=1)
    contours, _ = cv2.findContours(bw, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    contour_img = np.full_like(gray, 255)
    cv2.drawContours(contour_img, contours, -1, 0, 1, cv2.LINE_8)
    edges = 255 - contour_img
    return edges, len(contours)

# 2. ??? ??: Sobel/Laplacian/Canny? ????? ????, ?? ??? ?? ?? contour? ?????.
(sobel_raw, sobel_time) = timed_call(sobel_edges, kmeans_img, 65)
(lap_raw, lap_time) = timed_call(laplacian_edges, kmeans_img, 25)
(canny_raw, canny_time) = timed_call(canny_edges, kmeans_img, CANNY_LOW, CANNY_HIGH)

# ??? Hybrid: Lab ? ??? ? ??? Canny? ?????.
hybrid_start = time.perf_counter()
color_raw = color_boundary_edges(kmeans_img, min_delta=COLOR_EDGE_DELTA)
canny_covered = cv2.dilate(canny_raw, np.ones((3, 3), np.uint8), iterations=1)
color_new = cv2.bitwise_and(color_raw, cv2.bitwise_not(canny_covered))
hybrid_raw = cv2.bitwise_or(canny_raw, color_new)
hybrid_time = time.perf_counter() - hybrid_start

# ?? Object-first edge: ? ?? ?? ??? ?? ?? ??? ??, ? ???? ????.
object_start = time.perf_counter()
object_raw = object_first_edges(
    kmeans_labels,
    min_area=OBJECT_MIN_AREA_FOR_OUTPUT,
    close_kernel=OBJECT_CLOSE_KERNEL,
    thickness=LINE_THICKNESS,
)
dark_region_edges = dark_detail_region_edges(
    image,
    dark_threshold=DARK_DETAIL_THRESHOLD,
    min_area=SEGMENTATION_DARK_REGION_MIN_AREA_FOR_OUTPUT,
    max_area=SEGMENTATION_DARK_REGION_MAX_AREA_FOR_OUTPUT,
    thickness=LINE_THICKNESS,
)
source_boundary_edges = source_detail_boundary_edges(
    simplified_image,
    object_raw,
    low=SEGMENTATION_DETAIL_CANNY_LOW,
    high=SEGMENTATION_DETAIL_CANNY_HIGH,
    min_pixels=SEGMENTATION_DETAIL_MIN_PIXELS,
    max_pixels=SEGMENTATION_DETAIL_MAX_PIXELS,
    min_span=SEGMENTATION_DETAIL_MIN_SPAN,
    max_span=SEGMENTATION_DETAIL_MAX_SPAN,
    non_dark_threshold=SEGMENTATION_DETAIL_NON_DARK_THRESHOLD,
)
closed_detail_edges = closed_detail_shape_edges(
    image,
    object_raw,
    min_pixels=SEGMENTATION_CLOSED_DETAIL_MIN_PIXELS,
    max_pixels=SEGMENTATION_CLOSED_DETAIL_MAX_PIXELS,
    min_contour_area=SEGMENTATION_CLOSED_DETAIL_MIN_CONTOUR_AREA,
    max_contour_area=SEGMENTATION_CLOSED_DETAIL_MAX_CONTOUR_AREA,
    max_span=int(round(SEGMENTATION_CLOSED_DETAIL_MAX_SPAN * SEGMENTATION_OUTPUT_SCALE)),
)
segmentation_source_edges = cv2.bitwise_or(
    cv2.bitwise_or(cv2.bitwise_or(object_raw, dark_region_edges), source_boundary_edges),
    closed_detail_edges,
)
object_time = time.perf_counter() - object_start

sobel_clean = clean_edges(sobel_raw, open_iter=1, close_iter=1, thickness=1)
lap_clean = clean_edges(lap_raw, open_iter=1, close_iter=1, thickness=1)
canny_clean = clean_edges(canny_raw, open_iter=0, close_iter=1, thickness=2)
hybrid_compare_clean = clean_edges(hybrid_raw, open_iter=0, close_iter=1, thickness=EDGE_COMPARE_THICKNESS)
# ?? ???? ??? ???? ?? ????? ? ?? ??? ?? ?????.
# ?? ??? 1px contour? ????? ??? ??? ?? ?? ?????.
object_seg_clean = clean_edges(segmentation_source_edges, open_iter=0, close_iter=1, thickness=2)
object_final_clean = clean_edges(segmentation_source_edges, open_iter=0, close_iter=1, thickness=1)
detail_edges = detail_expression_edges(
    image,
    object_final_clean,
    low=DETAIL_CANNY_LOW,
    high=DETAIL_CANNY_HIGH,
    min_area=DETAIL_MIN_AREA_FOR_OUTPUT,
    max_area=DETAIL_MAX_AREA_FOR_OUTPUT,
    dark_threshold=DARK_DETAIL_THRESHOLD,
    dark_min_area=DARK_DETAIL_MIN_AREA_FOR_OUTPUT,
    dark_max_area=DARK_DETAIL_MAX_AREA_FOR_OUTPUT,
)
detail_line = coloring_line_image(detail_edges)

sobel_line = coloring_line_image(sobel_clean)
lap_line = coloring_line_image(lap_clean)
canny_line = coloring_line_image(canny_clean)
hybrid_compare_line = coloring_line_image(hybrid_compare_clean)
(adaptive_contour_raw, adaptive_contour_count), adaptive_contour_time = timed_call(adaptive_threshold_contour_edges, image)
adaptive_contour_clean = clean_edges(adaptive_contour_raw, open_iter=0, close_iter=1, thickness=1)
adaptive_contour_line = coloring_line_image(adaptive_contour_clean)
segmentation_line_image = coloring_line_image(object_seg_clean)
object_line = coloring_line_image(object_final_clean)
final_line_image = object_line

edge_rows = [
    {"algorithm": "Sobel", "runtime_sec": sobel_time, "edge_density": edge_density(sobel_clean), "hci_note": "??? ?? ??, ?? ??? ??"},
    {"algorithm": "Laplacian", "runtime_sec": lap_time, "edge_density": edge_density(lap_clean), "hci_note": "?? ?? ??, ?? ??? ??"},
    {"algorithm": "Canny", "runtime_sec": canny_time, "edge_density": edge_density(canny_clean), "hci_note": "???? ??? ?? ??"},
    {"algorithm": "Hybrid Color Boundary", "runtime_sec": hybrid_time, "edge_density": edge_density(hybrid_compare_clean), "hci_note": "? ?? ??? ??"},
    {"algorithm": "Adaptive Threshold Contour (02)", "runtime_sec": adaptive_contour_time, "edge_density": edge_density(adaptive_contour_clean), "hci_note": "빠르지만 색칠 영역 닫힘이 불안정해 비교용"},
    {"algorithm": "Object-first Contour", "runtime_sec": object_time, "edge_density": edge_density(object_final_clean), "hci_note": "?? ??? ?? ?? ??? ??"},
]

# 3. ?? ?? ? ???: 03? ???? ?? Connected Components ?? ??? ?????.
(region_map, regions), seg_time = timed_call(segment_connected_components, segmentation_line_image, MIN_REGION_AREA_FOR_OUTPUT)
(adaptive_region_map, adaptive_regions), adaptive_seg_time = timed_call(segment_connected_components, adaptive_contour_line, MIN_REGION_AREA_FOR_OUTPUT)
algorithm_compare_rows = [
    {
        "algorithm": "Adaptive Threshold Contour (02)",
        "edge_density": edge_density(adaptive_contour_clean),
        "regions": len(adaptive_regions),
        "runtime_sec": adaptive_contour_time + adaptive_seg_time,
        "decision": "comparison only",
    },
    {
        "algorithm": "Object-first Contour (current)",
        "edge_density": edge_density(object_seg_clean),
        "regions": len(regions),
        "runtime_sec": object_time + seg_time,
        "decision": "selected",
    },
]
background_color = estimate_background_color(kmeans_img)
regions = assign_region_color_numbers(
    regions,
    region_map,
    kmeans_labels,
    kmeans_palette,
    background_color=background_color,
    background_color_threshold=16,
    merge_background_similar=True,
)
background_region_ids = border_connected_region_ids(region_map)
for region in regions:
    if int(region["id"]) in background_region_ids:
        region["is_background"] = True
numbered_regions = [region for region in regions if not region.get("is_background", False)]
region_preview = color_region_preview(region_map)
color_edge_preview = color_region_edge_preview(final_line_image, region_map, regions, kmeans_palette, thickness=3)

# ? ??? dominant ??? ??? ?? ???????.
colored_by_labels = np.full_like(simplified_image, 255)
for region in regions:
    color = region.get("color_rgb")
    if color is None:
        continue
    colored_by_labels[region_map == region["id"]] = np.array(color, dtype=np.uint8)
colored_by_labels[segmentation_line_image < 128] = 0

line_with_detail = final_line_image.copy()
line_with_detail[detail_edges > 0] = 0
colored_by_labels_with_detail = colored_by_labels.copy()
colored_by_labels_with_detail[detail_edges > 0] = 0

# ?? ??? ??? ??? ?? ?? ??? ?? ?? ?? ?? ?? ??? ??????.
numbered_coloringbook = draw_paint_by_number_style(
    final_line_image,
    detail_edges,
    regions,
    region_map,
    background_region_ids,
    source_image=simplified_image,
    segmentation_edges=object_seg_clean,
)
segmentation_only_image = render_segmentation_only(region_map, regions)
detail_only_image = render_detail_only(detail_edges)
colored_by_labels_numbered = label_regions(
    colored_by_labels_with_detail,
    regions,
    font_scale=0.65,
    region_map=region_map,
    skip_background=True,
)
color_index = save_color_index_table(kmeans_palette, f"{OUTPUT_DIR}/04_color_index_table.png", regions, "Final Color Index")
numbered_with_index = combine_with_color_index(
    numbered_coloringbook,
    kmeans_palette,
    regions,
    f"{OUTPUT_DIR}/04_numbered_with_color_index.png",
    "Final Color Index",
)


def render_numbered_coloringbook_for_k(target_k):
    """Render a final numbered coloring-book image for one difficulty K."""
    (variant_img, variant_palette, variant_labels), variant_kmeans_time = timed_call(
        kmeans_quantization_with_labels,
        base_simplified_for_variants,
        target_k,
    )
    variant_source = base_image_for_variants.copy()
    variant_simplified = base_simplified_for_variants.copy()
    variant_source, variant_simplified, variant_img, variant_labels = upscale_for_segmentation_output(
        variant_source,
        variant_simplified,
        variant_img,
        variant_labels,
        variant_palette,
        SEGMENTATION_OUTPUT_SCALE,
    )
    variant_object_raw = object_first_edges(
        variant_labels,
        min_area=OBJECT_MIN_AREA_FOR_OUTPUT,
        close_kernel=OBJECT_CLOSE_KERNEL,
        thickness=LINE_THICKNESS,
    )
    variant_dark_region_edges = dark_detail_region_edges(
        variant_source,
        dark_threshold=DARK_DETAIL_THRESHOLD,
        min_area=SEGMENTATION_DARK_REGION_MIN_AREA_FOR_OUTPUT,
        max_area=SEGMENTATION_DARK_REGION_MAX_AREA_FOR_OUTPUT,
        thickness=LINE_THICKNESS,
    )
    variant_source_boundary_edges = source_detail_boundary_edges(
        variant_simplified,
        variant_object_raw,
        low=SEGMENTATION_DETAIL_CANNY_LOW,
        high=SEGMENTATION_DETAIL_CANNY_HIGH,
        min_pixels=SEGMENTATION_DETAIL_MIN_PIXELS,
        max_pixels=SEGMENTATION_DETAIL_MAX_PIXELS,
        min_span=SEGMENTATION_DETAIL_MIN_SPAN,
        max_span=SEGMENTATION_DETAIL_MAX_SPAN,
        non_dark_threshold=SEGMENTATION_DETAIL_NON_DARK_THRESHOLD,
    )
    variant_closed_detail_edges = closed_detail_shape_edges(
        variant_source,
        variant_object_raw,
        min_pixels=SEGMENTATION_CLOSED_DETAIL_MIN_PIXELS,
        max_pixels=SEGMENTATION_CLOSED_DETAIL_MAX_PIXELS,
        min_contour_area=SEGMENTATION_CLOSED_DETAIL_MIN_CONTOUR_AREA,
        max_contour_area=SEGMENTATION_CLOSED_DETAIL_MAX_CONTOUR_AREA,
        max_span=int(round(SEGMENTATION_CLOSED_DETAIL_MAX_SPAN * SEGMENTATION_OUTPUT_SCALE)),
    )
    variant_segmentation_source_edges = cv2.bitwise_or(
        cv2.bitwise_or(cv2.bitwise_or(variant_object_raw, variant_dark_region_edges), variant_source_boundary_edges),
        variant_closed_detail_edges,
    )
    variant_object_seg_clean = clean_edges(variant_segmentation_source_edges, open_iter=0, close_iter=1, thickness=2)
    variant_object_final_clean = clean_edges(variant_segmentation_source_edges, open_iter=0, close_iter=1, thickness=1)
    variant_detail_edges = detail_expression_edges(
        variant_source,
        variant_object_final_clean,
        low=DETAIL_CANNY_LOW,
        high=DETAIL_CANNY_HIGH,
        min_area=DETAIL_MIN_AREA_FOR_OUTPUT,
        max_area=DETAIL_MAX_AREA_FOR_OUTPUT,
        dark_threshold=DARK_DETAIL_THRESHOLD,
        dark_min_area=DARK_DETAIL_MIN_AREA_FOR_OUTPUT,
        dark_max_area=DARK_DETAIL_MAX_AREA_FOR_OUTPUT,
    )
    variant_segmentation_line_image = coloring_line_image(variant_object_seg_clean)
    variant_final_line_image = coloring_line_image(variant_object_final_clean)
    (variant_region_map, variant_regions), variant_seg_time = timed_call(
        segment_connected_components,
        variant_segmentation_line_image,
        MIN_REGION_AREA_FOR_OUTPUT,
    )
    variant_background_color = estimate_background_color(variant_img)
    variant_regions = assign_region_color_numbers(
        variant_regions,
        variant_region_map,
        variant_labels,
        variant_palette,
        background_color=variant_background_color,
        background_color_threshold=16,
        merge_background_similar=True,
    )
    variant_background_region_ids = border_connected_region_ids(variant_region_map)
    for variant_region in variant_regions:
        if int(variant_region["id"]) in variant_background_region_ids:
            variant_region["is_background"] = True
    variant_numbered = draw_paint_by_number_style(
        variant_final_line_image,
        variant_detail_edges,
        variant_regions,
        variant_region_map,
        variant_background_region_ids,
        source_image=variant_simplified,
        segmentation_edges=variant_object_seg_clean,
    )
    variant_numbered_regions = [r for r in variant_regions if not r.get("is_background", False)]
    variant_metrics = region_metrics(variant_numbered_regions, variant_source.shape, small_area=300)
    return variant_numbered, {
        "K": target_k,
        "regions": variant_metrics["regions"],
        "small_regions": variant_metrics["small_regions"],
        "kmeans_time": variant_kmeans_time,
        "seg_time": variant_seg_time,
    }


difficulty_file_names = {
    "\uC26C\uC6C0": "easy",
    "\uBCF4\uD1B5": "normal",
    "\uC5B4\uB824\uC6C0": "hard",
}
difficulty_result_images = []
difficulty_result_rows = []
for difficulty_name, difficulty_k in DIFFICULTY_K_VALUES.items():
    variant_numbered, variant_info = render_numbered_coloringbook_for_k(difficulty_k)
    file_key = difficulty_file_names.get(difficulty_name, str(difficulty_name))
    variant_path = f"{OUTPUT_DIR}/04_difficulty_{file_key}_k{difficulty_k}.png"
    save_image_rgb(variant_path, variant_numbered)
    difficulty_result_images.append((f"{difficulty_name} (K={difficulty_k})", variant_numbered))
    difficulty_result_rows.append({"difficulty": difficulty_name, **variant_info, "path": variant_path})

show_images(
    difficulty_result_images,
    cols=3,
    figsize=(12, 6),
    cmap="gray",
    save_path=f"{OUTPUT_DIR}/04_difficulty_comparison.png",
)

# Contour? Watershed? 03?? ?? ??? ??? ?? ?????.
contour_preview, contours = contour_regions(segmentation_line_image, MIN_REGION_AREA_FOR_OUTPUT)
watershed_preview, watershed_markers = watershed_segmentation(kmeans_img)

# 4. ??: ??? ??? ?? ??? ?????.
save_image_rgb(f"{OUTPUT_DIR}/04_original.png", image)
save_image_rgb(f"{OUTPUT_DIR}/04_simplified_input.png", simplified_image)
save_image_rgb(f"{OUTPUT_DIR}/04_kmeans.png", kmeans_img)
save_image_rgb(f"{OUTPUT_DIR}/04_posterization.png", poster_img)
save_image_rgb(f"{OUTPUT_DIR}/04_median_cut.png", median_img)
save_image_rgb(f"{OUTPUT_DIR}/04_sobel.png", sobel_line)
save_image_rgb(f"{OUTPUT_DIR}/04_laplacian.png", lap_line)
save_image_rgb(f"{OUTPUT_DIR}/04_canny.png", canny_line)
save_image_rgb(f"{OUTPUT_DIR}/04_hybrid_color_boundary.png", hybrid_compare_line)
save_image_rgb(f"{OUTPUT_DIR}/04_adaptive_threshold_contour_02.png", adaptive_contour_line)
save_image_rgb(f"{OUTPUT_DIR}/04_object_first_edges.png", object_line)
save_image_rgb(f"{OUTPUT_DIR}/04_segmentation_line_image.png", segmentation_line_image)
save_image_rgb(f"{OUTPUT_DIR}/04_segmentation_only.png", segmentation_only_image)
save_image_rgb(f"{OUTPUT_DIR}/04_detail_edges.png", detail_line)
save_image_rgb(f"{OUTPUT_DIR}/04_detail_only.png", detail_only_image)
save_image_rgb(f"{OUTPUT_DIR}/04_line_with_detail.png", line_with_detail)
save_image_rgb(f"{OUTPUT_DIR}/04_line_image.png", final_line_image)
save_image_rgb(f"{OUTPUT_DIR}/04_region_preview.png", region_preview)
save_image_rgb(f"{OUTPUT_DIR}/04_color_edge_preview.png", color_edge_preview)
save_image_rgb(f"{OUTPUT_DIR}/04_contour_preview.png", contour_preview)
save_image_rgb(f"{OUTPUT_DIR}/04_watershed_preview.png", watershed_preview)
save_image_rgb(f"{OUTPUT_DIR}/04_colored_by_labels.png", colored_by_labels)
save_image_rgb(f"{OUTPUT_DIR}/04_colored_by_labels_numbered.png", colored_by_labels_numbered)
save_image_rgb(f"{OUTPUT_DIR}/04_final_numbered_coloringbook.png", numbered_coloringbook)

show_images([
    ("1. Original", image),
    ("2. Simplified Input", simplified_image),
    ("3. K-Means", kmeans_img),
    ("4. Posterization", poster_img),
    ("5. Median Cut", median_img),
    ("6. Sobel", sobel_line),
    ("7. Laplacian", lap_line),
    ("8. Canny", canny_line),
    ("9. Hybrid Canny + Color Boundary", hybrid_compare_line),
    ("10. Adaptive Threshold Contour (02)", adaptive_contour_line),
    ("11. Thin Object-first Edge", object_line),
    ("12. Segmentation Edge", segmentation_line_image),
    ("13. Segmentation Only Layer", segmentation_only_image),
    ("14. Detail Only Layer", detail_only_image),
    ("15. Detail Expression Edge", detail_line),
    ("16. Line With Detail", line_with_detail),
    ("14. Segmentation", region_preview),
    ("15. Color Edge Preview", color_edge_preview),
    ("16. Contour Detection Preview", contour_preview),
    ("17. Watershed Preview", watershed_preview),
    ("18. Colored By Labels + Numbers", colored_by_labels_numbered),
    ("19. Final Numbered Coloring Book", numbered_coloringbook),
    ("20. Color Index Table", color_index),
    ("21. Difficulty Comparison", cv2.cvtColor(cv2.imread(f"{OUTPUT_DIR}/04_difficulty_comparison.png"), cv2.COLOR_BGR2RGB)),
], cols=2, figsize=(13, 32), cmap="gray", save_path=f"{OUTPUT_DIR}/04_final_results_grid.png")


## RGB ???


In [ ]:
plot_palette(kmeans_palette, "Final K-Means RGB Palette", f"{OUTPUT_DIR}/04_final_palette.png")


## ?? ?? ?


In [ ]:
metrics = region_metrics(numbered_regions, image.shape, small_area=300)
difficulty_rows = [
    {"difficulty": name, "K": value, "selected": name == DIFFICULTY}
    for name, value in DIFFICULTY_K_VALUES.items()
]
performance_rows = [
    {"stage": "Input Simplification", "runtime_sec": simplify_time, "color_error": "", "unique_colors": base_simplified_color_count, "edge_density": "", "regions": "", "average_area": "", "small_regions": ""},
    {"stage": "K-Means", "runtime_sec": kmeans_time, "color_error": quality_rows[0]["color_error"], "unique_colors": quality_rows[0]["unique_colors"], "edge_density": "", "regions": "", "average_area": "", "small_regions": ""},
    {"stage": "Posterization", "runtime_sec": poster_time, "color_error": quality_rows[1]["color_error"], "unique_colors": quality_rows[1]["unique_colors"], "edge_density": "", "regions": "", "average_area": "", "small_regions": ""},
    {"stage": "Median Cut", "runtime_sec": median_time, "color_error": quality_rows[2]["color_error"], "unique_colors": quality_rows[2]["unique_colors"], "edge_density": "", "regions": "", "average_area": "", "small_regions": ""},
    {"stage": "Sobel", "runtime_sec": sobel_time, "color_error": "", "unique_colors": "", "edge_density": edge_density(sobel_clean), "regions": "", "average_area": "", "small_regions": ""},
    {"stage": "Laplacian", "runtime_sec": lap_time, "color_error": "", "unique_colors": "", "edge_density": edge_density(lap_clean), "regions": "", "average_area": "", "small_regions": ""},
    {"stage": "Canny", "runtime_sec": canny_time, "color_error": "", "unique_colors": "", "edge_density": edge_density(canny_clean), "regions": "", "average_area": "", "small_regions": ""},
    {"stage": "Hybrid Color Boundary", "runtime_sec": hybrid_time, "color_error": "", "unique_colors": "", "edge_density": edge_density(hybrid_compare_clean), "regions": "", "average_area": "", "small_regions": ""},
    {"stage": "Adaptive Threshold Contour (02)", "runtime_sec": adaptive_contour_time + adaptive_seg_time, "color_error": "", "unique_colors": "", "edge_density": edge_density(adaptive_contour_clean), "regions": len(adaptive_regions), "average_area": "", "small_regions": ""},
    {"stage": "Object-first Contour", "runtime_sec": object_time, "color_error": "", "unique_colors": "", "edge_density": edge_density(object_final_clean), "regions": "", "average_area": "", "small_regions": ""},
    {"stage": "Detail Expression Edge", "runtime_sec": "", "color_error": "", "unique_colors": "", "edge_density": edge_density(detail_edges), "regions": "", "average_area": "", "small_regions": ""},
    {"stage": "Segmentation", "runtime_sec": seg_time, "color_error": "", "unique_colors": "", "edge_density": "", "regions": metrics["regions"], "average_area": metrics["average_area"], "small_regions": metrics["small_regions"]},
    {"stage": "Background-like Regions", "runtime_sec": "", "color_error": "", "unique_colors": "", "edge_density": "", "regions": sum(1 for region in regions if region.get("is_background")), "average_area": "", "small_regions": ""},
    {"stage": "Total Final", "runtime_sec": simplify_time + kmeans_time + object_time + seg_time, "color_error": "", "unique_colors": "", "edge_density": edge_density(object_final_clean), "regions": metrics["regions"], "average_area": metrics["average_area"], "small_regions": metrics["small_regions"]},
]

print(f"Input image: {IMAGE_PATH}  (original colors: {count_unique_colors(image)}, smoothed colors: {count_unique_colors(simplified_image)})")
print(f"Smoothed coarse color groups = {k_recommendation_info['coarse_unique_colors']}")
print(f"Recommended minimum K = {MIN_RECOMMENDED_K}")
print(f"Difficulty = {DIFFICULTY}, final K = {K}")
print(f"Output canvas = {output_scale_info['output_shape'][1]}x{output_scale_info['output_shape'][0]} (scale {output_scale_info['scale']}x)\n")
print("[Difficulty K]")
print_table(difficulty_rows, columns=["difficulty", "K", "selected"])
print("\n[Color Quantization Quality]")
print_table(quality_rows, columns=["algorithm", "runtime_sec", "color_error", "unique_colors"])
print("\n[Edge Detection]")
print_table(edge_rows, columns=["algorithm", "runtime_sec", "edge_density", "hci_note"])
print("\n[Algorithm Comparison For Segmentation]")
print_table(algorithm_compare_rows, columns=["algorithm", "edge_density", "regions", "runtime_sec", "decision"])
print("\n[Final Pipeline]")
print_table(performance_rows)


## ?? ??? ??? ??


In [ ]:
# ???? K? ????? ?? ?? ?? ??? ??? ???? ?????.
complexity_rows = complexity_by_k(simplified_image, k_values=K_VALUES, min_area=MIN_REGION_AREA)
print_table(complexity_rows)

ks = [row["K"] for row in complexity_rows]
regions_by_k = [row["regions"] for row in complexity_rows]
runtime_by_k = [row["runtime_sec"] for row in complexity_rows]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(ks, regions_by_k, marker="o")
axes[0].set_title("K vs Number of Regions")
axes[0].set_xlabel("K")
axes[0].set_ylabel("Regions")
axes[1].plot(ks, runtime_by_k, marker="o", color="tab:red")
axes[1].set_title("K vs Runtime")
axes[1].set_xlabel("K")
axes[1].set_ylabel("Seconds")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/04_complexity_by_k.png", dpi=160, bbox_inches="tight")
plt.show()


## ?? ??: ???? ???

### ?? ???

K-Means ??: ??? ?? ??? ??? ???? ???? ?? ?? ??? ????.  
K-Means ??: ?? ??? ??? Posterization?? ????.

Posterization ??: ?? ??? ??? ?????.  
Posterization ??: ?? ??? ?? ??? ???? ?? ????? ?????? ?? ??? ?? ? ????.

Median Cut ??: ?? ?? ??? ????? ???? ??? ??? ?? ??? ????.  
Median Cut ??: ??? ???? K-Means?? ??? ?? ??? ?? ?? ??? ???.

?? ??: `color_error`? ??? ??? ???? Lab ? ?? ???? ???? ?? ? ??? ????. `unique_colors`? ?? ???? ??? ?? ??? ????.

### ??? ??

Sobel ??: ??? ?? ?? ??? ?? ????.  
Sobel ??: ?? ???? ?? ??? ??? ? ????.

Laplacian ??: ?? ??? ??? ??? ? ?? ?????.  
Laplacian ??: ???? ?? ????? ??? ?? ??? ????.

Canny ??: ??? ??? ?? ???? ??? ???? ???? ??? ?????.  
Canny ??: ??? ??? ?? ?? ?? ??? ?? ? ????.

Hybrid Color Boundary ??: Canny ??? ?? ?? ??? ?? ?? ?? ??? ?????.  
Hybrid Color Boundary ??: K-Means ??? ????? ?? ??? ???? ??? ? ????.

Adaptive Threshold Contour(02번 추가 알고리즘) 장점: grayscale 기반이라 빠르고 구현이 단순하며 명암선 확인용으로 좋습니다.  
Adaptive Threshold Contour(02번 추가 알고리즘) 단점: 종이 질감, 그림자, 내부 명암에 민감하고 선이 닫히지 않아 Connected Components 영역 분리에는 불안정합니다.

### ?? ??

Connected Components ??: ?? ? ??? ?? ????, ? ??? dominant ?? ??? ?? ?? ?? ???? ?????.  
Connected Components ??: ?? ??? ??? ??? ? ?? Morphology ??? ?????.

?? ?? ?? ??: ???? ??? ???? ??? ?? ? ??? ?? ?? ??? ???? ???? ?????.

Contour Detection ??: ??? ???? ?? ?? ??? ????.  
Contour Detection ??: ?? ?? ?? ??? ??? ? ????.

Watershed ??: ?? ?? ??? ????.  
Watershed ??: ??????? ???? ?? ???? ???? ??? ?? ??? ? ? ????.

### ?? ???? ?? ??

최종 조합은 Pre-KMeans Smoothing + K-Means + Object-first Contour + Connected Components입니다. 02번에서 보낸 Adaptive Threshold Contour는 빠르고 선 비교용으로는 유용하지만, 현재 이미지에서는 닫힌 색칠 영역을 안정적으로 만들지 못해 최종 segmentation에는 사용하지 않습니다. Object-first Contour는 K-Means label map에서 색 객체를 먼저 만들고 외곽선을 그리기 때문에 영역 닫힘이 안정적이고, 색상 번호를 배치할 connected component를 더 잘 만듭니다. Hybrid Canny/Color Boundary는 비교용 결과로 유지합니다.

### Trade-off ??

- K ??: ??? ?????? ?? ?? ?? ?? ?? ?? ???? ?????.
- ? ?? ??: ?? ???? ????? ?? ??? ?? ?? ??? ??? ? ????.
- ?? ?? ?? ??: ?? ???? ????? ?? ??? ?????.
- Canny ??? ??: ???? ??? ??? ??? ??? ? ????.

### ??? ?? ???

- ?? ????? K=5, 10, 20, 50 ??? ???? ??? ??? ?????.
- K-Means, Posterization, Median Cut? ?? ??? ?? ??? ?? ?????.
- Sobel, Laplacian, Canny, Hybrid? Edge Density? ??? ???? ?????.
- ?? ??, ?? ?? ??, ?? ?? ??? HCI ??? ?????.
- ?? ??? K-Means ??? ??? ?? ??? ?????, ?? ?? ??? ?? ??? ?? ? ????.

### HCI ?? ???

- ?? ???? ???? ???? ? ?? ?? ?????.
- ?? ?/? ?? ??? ??? ??? ??? ???? ??????.
- ?? ?? ??? ?? ????? ?? ??? ?????.
- ?? ?? ?? ??? ?? ??? ?? ?? ???? ?????.
- ?? ?? ??? ???? ??? ???? ???? ????? ??? ??? ? ????.
